# 🕸️ PyG 그래프 신경망 다섯 모델 — Cora와 PROTEINS에서 표현력 비교하기

**건설빅데이터분석 · 그래프 신경망 · PyG GNN 다섯 모델 (Cora · PROTEINS) · 실습 노트북 (완성본)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baekja/Lecture_GNN-AE-Practice/blob/main/PyG_FiveGNN_PROTEINS/gnn5_practice.ipynb) ← **실습본**(빈칸 채우기) &nbsp;&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baekja/Lecture_GNN-AE-Practice/blob/main/PyG_FiveGNN_PROTEINS/gnn5_complete.ipynb) ← **완성본**(정답·실행결과 포함)

> 앞에서 배운 네 모델(**VanillaGNN · GCN · GAT · GraphSAGE**)에 **GIN**을 더해, 같은 설정으로 두 과제를 비교합니다.
> (1) Cora 논문 분류(노드 분류) (2) PROTEINS 효소 여부(그래프 분류).
> GIN 강의의 핵심은 **표현력은 구조를 가를 수 있는 상한**이지, 그 자체로 정확도가 아니라는 점입니다. Cora처럼 이웃의 평균 분야가 힌트인 과제와, PROTEINS처럼 그래프 전체 구조·크기가 답과 맞닿은 과제에서 순위가 어떻게 바뀌는지 봅니다.

## 📌 강의 중점
- Cora: 공개 분할(학습 140 · 검증 500 · 시험 1,000)에서 다섯 모델을 seed 0·1·2로 비교
- GIN의 로짓 헤드: MLP는 2층 이상(`Linear → BN → ReLU → Linear → ReLU`), **마지막은 그냥 Linear** (클래스 점수에 BN/ReLU를 걸지 않음)
- PROTEINS: `seed 11`로 섞은 뒤 80/10/10 → **890 / 111 / 112**, 정확도는 **그래프 단위**
- 리드아웃: GCN 등은 `global_mean_pool`, GIN은 층별 `global_add_pool`을 이어 붙임. 같은 GCN에 sum 풀링(`GCN_sum`)을 넣어 리드아웃 효과를 따로 봄
- 다수 클래스 기준선부터 본다. 강의노트의 전체 표는 **seed 10개**이고, 이 노트북은 Colab CPU를 위해 **seed 3개**만 돌린다

## 🎯 학습 목표
1. 노드 분류와 그래프 분류에서 학습 루프·채점 단위가 어떻게 다른지 설명할 수 있다.
2. GIN 식 $h'_i = \mathrm{MLP}((1+\epsilon)h_i + \sum_j h_j)$ 를 코드로 옮기고, 로짓에 비선형이 들어가면 안 되는 이유를 말할 수 있다.
3. 같은 다섯 모델의 순위가 Cora와 PROTEINS에서 어떻게 다른지, 표현력과 정확도를 구분해 해석할 수 있다.


## 📋 이 노트북 사용법

| 표시 | 뜻 |
|:-:|---|
| ▶ | 셀 실행 — 셀을 클릭하고 `Shift + Enter` |
| ✏️ | 실습본에서 빈칸(`_________`)을 채우는 곳 |
| ✅ / ❌ | 자가검증 결과 — ❌ 가 나오면 힌트를 읽고 고쳐 보세요 |
| 🔸 | 실험 결과 안내 (오류가 아님) |

**주석(`#` 뒤의 글) 읽는 법** — 실행에는 영향이 없는 설명입니다.
- `1️⃣2️⃣3️⃣4️⃣` 학습 루프 단계 · `💡` 강의노트 연결 · `📐` 텐서 모양 · `⚠️` 흔한 실수 · `✅` 기대 결과 · `⏱` 걸린 시간

**Colab에서 처음 열었다면**
1. 메뉴 **파일 → Drive에 사본 저장** 을 먼저 누르세요.
2. 첫 코드 셀이 PyTorch Geometric(PyG)이 없으면 설치합니다. **CPU 런타임 그대로** 실행됩니다. Colab CPU에서 노트북 전체는 **약 8~12분** 걸립니다 (배정된 컴퓨터에 따라 더 걸릴 수 있습니다). seed 3개로 학습하는 셀이 대부분입니다.
3. 강의노트에 실은 전체 비교표는 seed **10개** CPU 실험입니다. 여기 숫자는 seed 0·1·2라 소수점과 순위 일부가 조금 다를 수 있지만, 해석의 방향(Cora에서는 GCN이 앞서고, PROTEINS에서는 GIN이 앞선다)은 같습니다.
4. 위에서부터 **차례대로** 실행하세요.


---
## 0. 준비 — PyG 설치와 라이브러리


In [1]:
import importlib.util, subprocess, sys                 # 설치 확인 · 명령 실행 도구
if importlib.util.find_spec("torch_geometric") is None:  # PyG 가 없으면 (Colab 처음 실행)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"], check=True)  # 이 커널에 설치
print("PyG 준비 완료")                                 # 이미 있으면 바로 여기로


PyG 준비 완료


In [2]:
import time                                            # 걸린 시간 재기 (⏱)
import torch                                           # PyTorch: 텐서와 신경망
from torch import nn                                   # nn = 신경망 부품(층) 모음
import torch.nn.functional as F                        # F = 렐루 · 드롭아웃 · 손실 함수
import torch_geometric                                 # PyG 본체 (seed 고정 함수)
from torch_geometric.datasets import Planetoid, TUDataset  # Cora · PROTEINS
Planetoid.url = "https://raw.githubusercontent.com/kimiyoung/planetoid/master/data"  # ⚠️ 받을 주소
from torch_geometric.nn import (                       # PyG 가 만들어 둔 층과 풀링
    GCNConv, GATConv, SAGEConv, GINConv,
    global_mean_pool, global_add_pool,
)
from torch_geometric.loader import DataLoader          # 그래프를 묶어 배치로
from torch_geometric.utils import to_dense_adj         # VanillaGNN 용 밀집 인접행렬
import pandas as pd                                    # 결과 표

torch.set_printoptions(precision=3, sci_mode=False)    # 소수 3자리, 지수 표기 끄기
print("PyTorch", torch.__version__, "· PyG", torch_geometric.__version__)  # 버전 확인


/tmp/colab_mimic/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch 2.9.1+cpu · PyG 2.8.1


### 자가검증 도우미 `check()` · `check_range()` 준비
아래 셀은 결과가 맞는지 ✅ / ❌ 로 알려 주는 **도우미 함수**입니다.


In [3]:
def check(name, value, expected, tol=0.01):  # 자가검증 (이름, 결과, 기대값, 허용오차)
    """value 가 expected 와 tol 이내로 같으면 ✅, 다르면 ❌ 를 보여 줍니다."""
    value = torch.as_tensor(value, dtype=torch.float32)        # 숫자·리스트 → 텐서
    expected = torch.as_tensor(expected, dtype=torch.float32)  # 기대값도 텐서로
    if value.shape != expected.shape:                          # ⚠️ 모양부터 다르면
        print(f"❌ {name}: 모양 {tuple(value.shape)} ≠ 기대 {tuple(expected.shape)}")
        return                                                 # 여기서 끝
    gap = (value - expected).abs().max().item()                # 가장 큰 차이
    mark = "✅" if gap <= tol else "❌"                         # 허용오차 안이면 통과
    tol_s = f"{tol:.6f}".rstrip("0").rstrip(".")               # 0.000001 처럼 보기 좋게
    print(f"{mark} {name}: 최대 차이 {gap:.6f} (허용 {tol_s})")  # 결과 출력

def check_range(name, value, lo, hi):        # 자가검증 (이름, 결과, 최솟값, 최댓값)
    """value 가 lo ~ hi 범위 안이면 ✅, 벗어나면 ❌ 를 보여 줍니다."""
    value = float(value)                                       # 텐서·숫자 → 파이썬 실수
    mark = "✅" if lo <= value <= hi else "❌"                  # 범위 안이면 통과
    print(f"{mark} {name}: {value:.4f} (기대 범위 {lo} ~ {hi})")  # 결과 출력


---
## 1. Cora — 노드 분류 (공개 분할)

Cora는 논문 2,708편이 인용으로 이어진 **그래프 하나**입니다. 각 노드(논문)의 분야(7개)를 맞힙니다.
정답을 아는 학습 노드는 분야마다 20개, 모두 **140개**뿐입니다(준지도 노드 분류).


In [4]:
dataset = Planetoid(root="data", name="Cora")          # Cora 내려받기
data = dataset[0]                                      # 그래프가 한 개뿐 → Data 객체 하나
print(data)                                            # 들어 있는 텐서와 모양 📐
print("학습 · 검증 · 시험 노드:", int(data.train_mask.sum()),
      int(data.val_mask.sum()), int(data.test_mask.sum()))
print("분야 수:", dataset.num_classes, "· 피처:", dataset.num_features)
y_test = data.y[data.test_mask]                        # 시험 노드 1,000개의 정답
test_maj = int(torch.bincount(y_test).max()) / len(y_test) * 100  # 다수 클래스만 찍기
print(f"시험셋 다수 클래스 기준선: {test_maj:.2f}%")     # 💡 의미 있는 더미 기준선
print(f"균등 추측 기준선: {100 / dataset.num_classes:.2f}%")  # 학습셋은 클래스당 20개로 동률
check("Cora 시험 노드 수", int(data.test_mask.sum()), 1000, tol=0)
check("Cora 시험셋 다수 클래스 기준선", test_maj, 31.90, tol=0.05)


Processing...


Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
학습 · 검증 · 시험 노드: 140 500 1000
분야 수: 7 · 피처: 1433
시험셋 다수 클래스 기준선: 31.90%
균등 추측 기준선: 14.29%
✅ Cora 시험 노드 수: 최대 차이 0.000000 (허용 0)
✅ Cora 시험셋 다수 클래스 기준선: 최대 차이 0.000000 (허용 0.05)


Done!


🔍 **결과 읽는 법** — `x=[2708, 1433]` 은 노드 2,708개 × 단어 피처 1,433개입니다.
시험셋에서 가장 흔한 분야만 찍어도 약 **31.9%** 입니다. 학습셋은 분야마다 20개로 같아서, "학습셋 다수 클래스"는 동률을 깨는 인위적인 값이 됩니다. 해석에는 시험셋 다수 클래스(31.9%)와 균등 추측(약 14.3%)을 씁니다.


---
## 2. Cora 학습 · 시험 함수 — 다섯 모델이 함께 씁니다

손실은 **학습 마스크가 참인 노드만**으로 계산합니다. 시험 노드의 정답을 보면 정보가 새어 나갑니다.
모델은 **로짓(점수)** 을 내고, 손실은 `cross_entropy`가 안에서 로그소프트맥스를 적용합니다.


In [5]:
def train_cora(model, inputs, epochs=200):             # 모델 학습 (그래프 전체를 한 번에)
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)  # 아담
    mask = data.train_mask                             # 학습 노드 140개
    for epoch in range(epochs):                        # 200번 반복
        model.train()                                  # 학습 모드 (드롭아웃 켜기)
        out = model(*inputs)                           # 1️⃣ 순전파: 모든 노드의 점수 📐 [2708, 7]
        loss = F.cross_entropy(out[mask], data.y[mask])  # 2️⃣ 학습 노드만
        opt.zero_grad()                                # 3️⃣ 기울기 비우고
        loss.backward()                                #    역전파로 기울기 계산
        opt.step()                                     # 4️⃣ 가중치 업데이트
    return model                                       # 학습된 모델

def test_cora(model, inputs):                          # 시험 정확도(%)를 돌려주는 함수
    model.eval()                                       # 평가 모드 (드롭아웃 끄기)
    with torch.no_grad():                              # 평가에는 기울기가 필요 없음
        pred = model(*inputs).argmax(dim=1)            # 점수가 가장 큰 분야
    mask = data.test_mask                              # 시험 노드 1,000개
    return (pred[mask] == data.y[mask]).float().mean().item() * 100  # 맞힌 비율 × 100

def three_seeds(make_model, train_in, test_in=None, train_fn=train_cora, test_fn=test_cora):
    t0, accs = time.time(), []                         # 시작 시각, 정확도를 모을 리스트
    for seed in (0, 1, 2):                             # 초기 가중치만 바꿔 세 번
        torch_geometric.seed_everything(seed)          # 같은 seed → 같은 결과
        model = train_fn(make_model(), train_in)       # 새 모델을 만들어 학습
        accs.append(test_fn(model, test_in or train_in))  # 시험
    accs = torch.tensor(accs)                          # 리스트 → 텐서 📐 [3]
    sec = time.time() - t0                             # ⏱ 세 번 학습에 걸린 시간
    print("seed별", [round(a, 1) for a in accs.tolist()], f"⏱ {sec:.0f}초")
    print(f"→ 시험 정확도 {accs.mean():.1f} ± {accs.std():.1f} %")
    return accs


---
## 3. 네 모델 — VanillaGNN · GCN · GAT · GraphSAGE

뼈대는 같습니다: 층 두 개, 은닉 16(GAT는 헤드 8 × 8차원), 200 에포크, Adam `lr=0.01`, 가중치 감쇠 `5e-4`.
차이는 **이웃을 어떻게 모으느냐**입니다. VanillaGNN은 $\tilde A = A+I$ 로 더하고, GCN은 차수로 나누며, GAT는 이웃마다 다른 계수, GraphSAGE는 자기와 이웃 평균을 따로 둡니다.


In [6]:
class VanillaGNNLayer(nn.Module):                      # VanillaGNN 층 하나: Ã X Wᵀ
    def __init__(self, dim_in, dim_out):
        super().__init__()
        self.linear = nn.Linear(dim_in, dim_out, bias=False)  # 가중치 W (편향 없음)
    def forward(self, x, adj):                         # adj = Ã (자기 루프 포함 인접행렬)
        return adj @ self.linear(x)                    # ① 노드마다 x Wᵀ  ② 이웃 + 자기의 합

class VanillaGNN(nn.Module):
    def __init__(self, dim_in, dim_h, dim_out):
        super().__init__()
        self.gnn1 = VanillaGNNLayer(dim_in, dim_h)
        self.gnn2 = VanillaGNNLayer(dim_h, dim_out)
    def forward(self, x, adj):
        x = F.relu(self.gnn1(x, adj))                  # 💡 합 집계, 정규화 없음
        return self.gnn2(x, adj)

class GCN(nn.Module):
    def __init__(self, dim_in, dim_h, dim_out):
        super().__init__()
        self.gcn1 = GCNConv(dim_in, dim_h)
        self.gcn2 = GCNConv(dim_h, dim_out)
    def forward(self, x, edge_index):
        x = F.dropout(x, p=0.5, training=self.training)
        x = F.relu(self.gcn1(x, edge_index))
        x = F.dropout(x, p=0.5, training=self.training)
        return self.gcn2(x, edge_index)                # 로짓 (소프트맥스 안 함)

class GAT(nn.Module):
    def __init__(self, dim_in, dim_h, dim_out, heads=8):
        super().__init__()
        self.gat1 = GATConv(dim_in, dim_h, heads=heads, dropout=0.6)
        self.gat2 = GATConv(dim_h * heads, dim_out, heads=1, dropout=0.6)
    def forward(self, x, edge_index):
        x = F.dropout(x, p=0.6, training=self.training)
        x = F.elu(self.gat1(x, edge_index))
        x = F.dropout(x, p=0.6, training=self.training)
        return self.gat2(x, edge_index)

class GraphSAGE(nn.Module):
    def __init__(self, dim_in, dim_h, dim_out):
        super().__init__()
        self.sage1 = SAGEConv(dim_in, dim_h, aggr="mean")
        self.sage2 = SAGEConv(dim_h, dim_out, aggr="mean")
    def forward(self, x, edge_index):
        x = F.relu(self.sage1(x, edge_index))
        x = F.dropout(x, p=0.5, training=self.training)
        return self.sage2(x, edge_index)

F_IN, C_OUT = dataset.num_features, dataset.num_classes
N = data.num_nodes
A_tilde = to_dense_adj(data.edge_index, max_num_nodes=N)[0] + torch.eye(N)  # Ã = A + I
results_cora = {}
results_cora["Vanilla"] = three_seeds(lambda: VanillaGNN(F_IN, 16, C_OUT), (data.x, A_tilde))
check_range("Cora Vanilla 평균", results_cora["Vanilla"].mean(), 68, 80)
results_cora["GCN"] = three_seeds(lambda: GCN(F_IN, 16, C_OUT), (data.x, data.edge_index))
check_range("Cora GCN 평균", results_cora["GCN"].mean(), 76, 84)
results_cora["GAT"] = three_seeds(lambda: GAT(F_IN, 8, C_OUT), (data.x, data.edge_index))
check_range("Cora GAT 평균", results_cora["GAT"].mean(), 75, 84)
results_cora["GraphSAGE"] = three_seeds(lambda: GraphSAGE(F_IN, 16, C_OUT), (data.x, data.edge_index))
check_range("Cora GraphSAGE 평균", results_cora["GraphSAGE"].mean(), 74, 84)


seed별 [75.7, 70.1, 75.1] ⏱ 6초
→ 시험 정확도 73.6 ± 3.1 %
✅ Cora Vanilla 평균: 73.6333 (기대 범위 68 ~ 80)


seed별 [80.3, 80.9, 80.2] ⏱ 42초
→ 시험 정확도 80.5 ± 0.4 %
✅ Cora GCN 평균: 80.4667 (기대 범위 76 ~ 84)


seed별 [79.3, 80.1, 78.1] ⏱ 48초
→ 시험 정확도 79.2 ± 1.0 %
✅ Cora GAT 평균: 79.1667 (기대 범위 75 ~ 84)


seed별 [80.1, 80.4, 79.9] ⏱ 14초
→ 시험 정확도 80.1 ± 0.3 %
✅ Cora GraphSAGE 평균: 80.1333 (기대 범위 74 ~ 84)


---
## 4. GIN — 합 집계 + 2층 MLP, 로짓은 Linear

GIN 한 층은 $h'_i = \mathrm{{MLP}}\big((1+\epsilon)h_i + \sum_{{j \in \mathcal{{N}}(i)}} h_j\big)$ 입니다.
- **합(sum)** 은 이웃 다중집합을 덜 접습니다 (mean · max 보다 표현력이 높음).
- **MLP는 2층 이상**이어야 단사에 가까운 매핑이 됩니다. 아래 `gin_mlp`가 그 자리입니다.
- $\epsilon = 0$ 으로 두면 자기와 이웃을 같은 비중으로 더합니다 (PyG 기본값).
- ⚠️ 마지막에 클래스 점수를 낼 때는 **BN과 ReLU를 걸지 않습니다.** 로짓이 음수일 수 있어야 하고, ReLU가 채널을 죽이기 때문입니다. 그래서 마지막은 `GINConv(은닉→은닉)` 다음에 `Linear(은닉→7)` 입니다.


In [7]:
def gin_mlp(a, b):                                     # GIN 안의 2층 MLP
    return nn.Sequential(                              # Linear → BN → ReLU → Linear → ReLU
        nn.Linear(a, b), nn.BatchNorm1d(b), nn.ReLU(),
        nn.Linear(b, b), nn.ReLU(),
    )

class GIN(nn.Module):                                  # Cora용 GIN (수정된 로짓 헤드)
    def __init__(self, dim_in, dim_h, dim_out):
        super().__init__()
        self.conv1 = GINConv(gin_mlp(dim_in, dim_h), eps=0.0, train_eps=False)
        self.conv2 = GINConv(gin_mlp(dim_h, dim_h), eps=0.0, train_eps=False)
        self.lin = nn.Linear(dim_h, dim_out)           # 로짓: BN/ReLU 없음
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)                  # MLP 가 이미 ReLU 를 포함
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return self.lin(x)                             # 분야별 점수 📐 [2708, 7]

results_cora["GIN"] = three_seeds(lambda: GIN(F_IN, 16, C_OUT), (data.x, data.edge_index))
check_range("Cora GIN 평균", results_cora["GIN"].mean(), 40, 72)


seed별 [58.8, 60.8, 56.5] ⏱ 12초
→ 시험 정확도 58.7 ± 2.2 %
✅ Cora GIN 평균: 58.7000 (기대 범위 40 ~ 72)


🔍 **결과 읽는 법** — Cora에서는 GCN · GAT · GraphSAGE가 서로 비슷하고 VanillaGNN도 그보다 조금 낮을 뿐입니다. GIN은 표현력은 더 높은데 시험 정확도는 대체로 낮습니다.
💡 이유는 "표현력이 모자라서"가 아닙니다. Cora는 **동질성**이 커서 이웃을 평균 내는 쪽이 잘 맞습니다. 정규화가 표현력을 깎아도 정확도가 오른다는 강의노트의 아이러니와 같은 이야기입니다.
⚠️ 만약 마지막 MLP를 `gin_mlp(은닉, 7)` 로 두면 클래스 채널에 BN과 ReLU가 걸려 점수가 더 크게 흔들립니다. 그 숫자는 GIN의 표현력 실험으로 인용하면 안 됩니다.


In [8]:
fmt = lambda r: f"{r.mean():.1f} ± {r.std():.1f}"
cora_table = pd.DataFrame({
    "모델": list(results_cora) + ["기준선 (시험셋 다수 클래스)"],
    "seed 0·1·2 시험 정확도(%)": [fmt(results_cora[k]) for k in results_cora] + [f"{test_maj:.1f}"],
})
print(cora_table.to_string(index=False))
print("🔸 강의노트 전체 표는 seed 10개입니다. Cora GCN 약 80.2, GIN(수정 헤드) 약 55.9, 시험셋 기준선 31.9.")


              모델 seed 0·1·2 시험 정확도(%)
         Vanilla           73.6 ± 3.1
             GCN           80.5 ± 0.4
             GAT           79.2 ± 1.0
       GraphSAGE           80.1 ± 0.3
             GIN           58.7 ± 2.2
기준선 (시험셋 다수 클래스)                 31.9
🔸 강의노트 전체 표는 seed 10개입니다. Cora GCN 약 80.2, GIN(수정 헤드) 약 55.9, 시험셋 기준선 31.9.


---
## 5. PROTEINS — 그래프 분류

PROTEINS는 단백질 **1,113개**, 단백질 하나가 그래프 하나입니다. 노드는 2차 구조 단위(나선·병풍·회전, 피처 3차원)이고, 맞힐 것은 그 단백질이 **효소인지**입니다.
그래프 전체를 하나의 벡터로 모으는 **리드아웃(readout)** 이 꼭 필요합니다.
강의노트와 같이 `seed 11`로 섞은 뒤 80/10/10으로 자릅니다 → **890 / 111 / 112**.


In [9]:
torch.manual_seed(11)                                  # 💡 분할을 고정 (노트 §2.1)
proteins = TUDataset(root="data", name="PROTEINS").shuffle()
print(f"그래프 {len(proteins)} · 노드 피처 {proteins.num_features}차원 · 클래스 {proteins.num_classes}")
print(f"평균 노드 수 {sum(g.num_nodes for g in proteins) / len(proteins):.1f}")
counts_all = torch.bincount(proteins.y)
print(f"전체 클래스 수 {counts_all.tolist()} → 다수 클래스 {counts_all.max() / len(proteins) * 100:.2f}%")

n = len(proteins)
train_p = proteins[:int(n * 0.8)]                      # 890
val_p   = proteins[int(n * 0.8):int(n * 0.9)]          # 111
test_p  = proteins[int(n * 0.9):]                      # 112
print(f"분할 {len(train_p)} / {len(val_p)} / {len(test_p)}")
counts_test = torch.bincount(torch.cat([g.y for g in test_p]))
test_maj_p = counts_test.max().item() / len(test_p) * 100
print(f"테스트셋 클래스 {counts_test.tolist()} → 다수 클래스 {test_maj_p:.2f}%")
check("PROTEINS 분할", [len(train_p), len(val_p), len(test_p)], [890, 111, 112], tol=0)
check("PROTEINS 테스트 다수 클래스", test_maj_p, 56.25, tol=0.05)

train_loader = DataLoader(train_p, batch_size=64, shuffle=True)   # 학습만 섞음
val_loader   = DataLoader(val_p, batch_size=64)        # 평가는 섞지 않음
test_loader  = DataLoader(test_p, batch_size=64)


Processing...


그래프 1113 · 노드 피처 3차원 · 클래스 2
평균 노드 수 39.1
전체 클래스 수 [663, 450] → 다수 클래스 59.57%
분할 890 / 111 / 112
테스트셋 클래스 [63, 49] → 다수 클래스 56.25%
✅ PROTEINS 분할: 최대 차이 0.000000 (허용 0)
✅ PROTEINS 테스트 다수 클래스: 최대 차이 0.000000 (허용 0.05)


Done!


⚠️ 정확도는 **맞힌 그래프 수 / 전체 그래프 수**로 셉니다. 배치 정확도를 배치 수로 나누면 마지막 배치(48개)와 앞 배치(64개)의 무게가 달라집니다 (강의노트 §1.10).
검증 로더는 만들만 하고, 이 노트북도 강의노트처럼 **마지막 에포크의 테스트 점수**를 보고합니다 (검증으로 모델을 고르지 않음).


In [10]:
def train_graphs(model, loader, epochs=100):           # 그래프 분류 학습
    opt = torch.optim.Adam(model.parameters(), lr=0.01)  # 💡 노트: wd 없음
    for epoch in range(epochs + 1):                    # 0 ~ 100, 101 step (노트와 같음)
        model.train()
        for batch in loader:                           # 배치마다
            opt.zero_grad()
            out = model(batch.x, batch.edge_index, batch.batch)
            loss = F.cross_entropy(out, batch.y)       # 그래프마다 한 줄
            loss.backward()
            opt.step()
    return model

@torch.no_grad()
def test_graphs(model, loader):                        # 맞힌 그래프 수 / 전체 그래프 수
    model.eval()
    correct = total = 0
    for batch in loader:
        pred = model(batch.x, batch.edge_index, batch.batch).argmax(dim=1)
        correct += int((pred == batch.y).sum())
        total += batch.num_graphs                      # ⚠️ 배치 수가 아니라 그래프 수
    return correct / total * 100

def three_seeds_graphs(make_model):
    t0, accs = time.time(), []
    for seed in (0, 1, 2):
        torch_geometric.seed_everything(seed)
        model = train_graphs(make_model(), train_loader)
        accs.append(test_graphs(model, test_loader))
    accs = torch.tensor(accs)
    sec = time.time() - t0
    print("seed별", [round(a, 1) for a in accs.tolist()], f"⏱ {sec:.0f}초")
    print(f"→ 시험 정확도 {accs.mean():.1f} ± {accs.std():.1f} %")
    return accs


### 5-1. mean 풀링 모델과 GIN, 그리고 GCN_sum

Vanilla · GCN · GAT · GraphSAGE는 마지막 노드 표현을 **평균**으로 모아 그래프 벡터를 만듭니다 (강의노트 GCN과 같은 헤드).
GIN은 매 층의 노드 표현을 **합**으로 모은 뒤 이어 붙입니다. 단백질 크기 정보가 남아 있습니다.
같은 GCN에 풀링만 합으로 바꾼 `GCN_sum`을 두면, GIN과 GCN의 차이 중 **얼마가 리드아웃 덕분인지** 가늠할 수 있습니다.


In [11]:
class GraphMeanNet(nn.Module):                         # 3층 + mean 풀링 (노트 GCN 헤드)
    def __init__(self, convs, dim_h, dim_out):
        super().__init__()
        self.conv1, self.conv2, self.conv3 = convs
        self.lin = nn.Linear(dim_h, dim_out)
    def forward(self, x, edge_index, batch):
        h = self.conv1(x, edge_index).relu()
        h = self.conv2(h, edge_index).relu()
        h = self.conv3(h, edge_index)
        hG = global_mean_pool(h, batch)                # 그래프 임베딩 = 노드 평균
        return self.lin(F.dropout(hG, p=0.5, training=self.training))

class VanillaGraph(GraphMeanNet):
    def __init__(self, dim_in, dim_h, dim_out):
        super().__init__((
            VanillaGNNLayer(dim_in, dim_h),
            VanillaGNNLayer(dim_h, dim_h),
            VanillaGNNLayer(dim_h, dim_h),
        ), dim_h, dim_out)
    def forward(self, x, edge_index, batch):           # edge_index → Ã (배치에서도 동작)
        # 배치된 여러 그래프를 밀집 행렬로 만들지 않고, (A+I) 합 집계를 직접 합니다
        h = x
        for i, conv in enumerate((self.conv1, self.conv2, self.conv3)):
            out = conv.linear(h)
            deg_like = torch.zeros_like(out)
            src, dst = edge_index
            deg_like.index_add_(0, dst, out[src])      # 이웃 합
            deg_like = deg_like + out                  # 자기 루프 (Ã = A + I)
            h = deg_like.relu() if i < 2 else deg_like
        hG = global_mean_pool(h, batch)
        return self.lin(F.dropout(hG, p=0.5, training=self.training))

class GCNGraph(nn.Module):
    def __init__(self, dim_in, dim_h, dim_out, pool="mean"):
        super().__init__()
        self.conv1 = GCNConv(dim_in, dim_h)
        self.conv2 = GCNConv(dim_h, dim_h)
        self.conv3 = GCNConv(dim_h, dim_h)
        self.lin = nn.Linear(dim_h, dim_out)
        self.pool = pool
    def forward(self, x, edge_index, batch):
        h = self.conv1(x, edge_index).relu()
        h = self.conv2(h, edge_index).relu()
        h = self.conv3(h, edge_index)
        pool = global_add_pool if self.pool == "sum" else global_mean_pool
        hG = pool(h, batch)
        return self.lin(F.dropout(hG, p=0.5, training=self.training))

class GATGraph(GraphMeanNet):
    def __init__(self, dim_in, dim_h, dim_out, heads=8):
        dh = dim_h // heads
        super().__init__((
            GATConv(dim_in, dh, heads=heads, concat=True),
            GATConv(dim_h, dh, heads=heads, concat=True),
            GATConv(dim_h, dim_h, heads=1, concat=False),
        ), dim_h, dim_out)

class SAGEGraph(GraphMeanNet):
    def __init__(self, dim_in, dim_h, dim_out):
        super().__init__((
            SAGEConv(dim_in, dim_h, aggr="mean"),
            SAGEConv(dim_h, dim_h, aggr="mean"),
            SAGEConv(dim_h, dim_h, aggr="mean"),
        ), dim_h, dim_out)

class GINGraph(nn.Module):                             # 노트 §2.3: 층별 sum + concat
    def __init__(self, dim_in, dim_h, dim_out):
        super().__init__()
        self.conv1 = GINConv(gin_mlp(dim_in, dim_h), eps=0.0, train_eps=False)
        self.conv2 = GINConv(gin_mlp(dim_h, dim_h), eps=0.0, train_eps=False)
        self.conv3 = GINConv(gin_mlp(dim_h, dim_h), eps=0.0, train_eps=False)
        self.lin1 = nn.Linear(dim_h * 3, dim_h * 3)
        self.lin2 = nn.Linear(dim_h * 3, dim_out)
    def forward(self, x, edge_index, batch):
        h1 = self.conv1(x, edge_index)
        h2 = self.conv2(h1, edge_index)
        h3 = self.conv3(h2, edge_index)
        h = torch.cat([                                 # 💡 모든 층의 sum 을 이어 붙임
            global_add_pool(h1, batch),
            global_add_pool(h2, batch),
            global_add_pool(h3, batch),
        ], dim=1)
        h = F.dropout(self.lin1(h).relu(), p=0.5, training=self.training)
        return self.lin2(h)                            # 로짓

P_IN, P_OUT = proteins.num_features, proteins.num_classes
results_prot = {}
results_prot["Vanilla"] = three_seeds_graphs(lambda: VanillaGraph(P_IN, 32, P_OUT))
check_range("PROTEINS Vanilla 평균", results_prot["Vanilla"].mean(), 50, 78)
results_prot["GCN"] = three_seeds_graphs(lambda: GCNGraph(P_IN, 32, P_OUT, pool="mean"))
check_range("PROTEINS GCN 평균", results_prot["GCN"].mean(), 62, 78)
results_prot["GAT"] = three_seeds_graphs(lambda: GATGraph(P_IN, 32, P_OUT))
check_range("PROTEINS GAT 평균", results_prot["GAT"].mean(), 58, 78)
results_prot["GraphSAGE"] = three_seeds_graphs(lambda: SAGEGraph(P_IN, 32, P_OUT))
check_range("PROTEINS GraphSAGE 평균", results_prot["GraphSAGE"].mean(), 56, 76)
results_prot["GIN"] = three_seeds_graphs(lambda: GINGraph(P_IN, 32, P_OUT))
check_range("PROTEINS GIN 평균", results_prot["GIN"].mean(), 66, 82)
results_prot["GCN_sum"] = three_seeds_graphs(lambda: GCNGraph(P_IN, 32, P_OUT, pool="sum"))
check_range("PROTEINS GCN_sum 평균", results_prot["GCN_sum"].mean(), 62, 82)


seed별 [65.2, 56.2, 65.2] ⏱ 34초
→ 시험 정확도 62.2 ± 5.2 %
✅ PROTEINS Vanilla 평균: 62.2024 (기대 범위 50 ~ 78)


seed별 [69.6, 67.0, 72.3] ⏱ 20초
→ 시험 정확도 69.6 ± 2.7 %
✅ PROTEINS GCN 평균: 69.6429 (기대 범위 62 ~ 78)


seed별 [67.9, 67.9, 69.6] ⏱ 53초
→ 시험 정확도 68.5 ± 1.0 %
✅ PROTEINS GAT 평균: 68.4524 (기대 범위 58 ~ 78)


seed별 [69.6, 67.9, 67.0] ⏱ 17초
→ 시험 정확도 68.2 ± 1.4 %
✅ PROTEINS GraphSAGE 평균: 68.1548 (기대 범위 56 ~ 76)


seed별 [75.9, 74.1, 71.4] ⏱ 22초
→ 시험 정확도 73.8 ± 2.2 %
✅ PROTEINS GIN 평균: 73.8095 (기대 범위 66 ~ 82)


seed별 [75.9, 69.6, 75.9] ⏱ 34초
→ 시험 정확도 73.8 ± 3.6 %
✅ PROTEINS GCN_sum 평균: 73.8095 (기대 범위 62 ~ 82)


In [12]:
prot_table = pd.DataFrame({
    "모델": list(results_prot) + ["기준선 (테스트 112개 다수 클래스)"],
    "seed 0·1·2 시험 정확도(%)": [fmt(results_prot[k]) for k in results_prot] + [f"{test_maj_p:.1f}"],
})
print(prot_table.to_string(index=False))
print("🔸 강의노트 seed 10개: GIN 73.21 ± 2.66, GCN_sum 71.34 ± 2.87, GCN 69.38 ± 1.79,")
print("   GAT 68.57 ± 2.05, GraphSAGE 66.61 ± 1.80, Vanilla 63.66 ± 4.21, 테스트 기준선 56.25.")
print("   GIN과 GCN의 차이(약 3.8%p) 중 일부는 sum 리드아웃(GCN_sum)만으로도 나옵니다.")


                   모델 seed 0·1·2 시험 정확도(%)
              Vanilla           62.2 ± 5.2
                  GCN           69.6 ± 2.7
                  GAT           68.5 ± 1.0
            GraphSAGE           68.2 ± 1.4
                  GIN           73.8 ± 2.2
              GCN_sum           73.8 ± 3.6
기준선 (테스트 112개 다수 클래스)                 56.2
🔸 강의노트 seed 10개: GIN 73.21 ± 2.66, GCN_sum 71.34 ± 2.87, GCN 69.38 ± 1.79,
   GAT 68.57 ± 2.05, GraphSAGE 66.61 ± 1.80, Vanilla 63.66 ± 4.21, 테스트 기준선 56.25.
   GIN과 GCN의 차이(약 3.8%p) 중 일부는 sum 리드아웃(GCN_sum)만으로도 나옵니다.


---
## 6. 종합 — 표현력은 정확도의 상한이지 정확도 자체가 아니다

| 관찰 | 이 실험에서 말할 수 있는 것 | 아직 말하면 안 되는 것 |
|---|---|---|
| Cora | 수정된 GIN도 GCN·GAT·SAGE보다 낮았다. 이웃을 평균 내는 모델이 이 과제에 잘 맞는다 | "GIN은 노드 분류에 항상 쓸모없다" (설정·정규화를 바꿔 보지 않았음) |
| PROTEINS | 이 분할에서 GIN이 GCN(mean)보다 높고, 모두 다수 클래스 기준선(56.25%)을 넘는다 | "GIN의 표현력 덕분"만으로 차이를 설명하기 (GCN_sum이 그 사이) |
| 기준선 | 숫자 해석은 기준선과 seed 분산부터 | seed 하나, 또는 기준선보다 낮은 값으로 이론을 말하기 |

💡 한 줄: **과제가 구조를 구별해야 할 때만 표현력이 정확도로 이어지기 쉽고**, 그 경우에도 리드아웃(mean vs sum)과 학습 안정성이 함께 움직입니다.
강의노트의 전체 표(seed 10개)를 공식 숫자로 쓰고, 이 노트북의 seed 3개는 같은 실험을 Colab에서 재현한 확인용입니다.


### ⚠️ 자주 하는 실수
1. Cora 손실을 **모든 노드**로 계산 → 시험 노드의 정답을 본 셈입니다. `out[train_mask]` 만!
2. PROTEINS 정확도를 배치 정확도의 평균으로 냄 → 마지막 배치 무게가 달라집니다. **맞힌 그래프 수 / 전체 그래프 수**.
3. GIN 마지막 MLP를 `gin_mlp(은닉, 클래스 수)`로 둠 → 로짓에 BN과 ReLU가 걸립니다. 마지막은 `Linear`.
4. 평가 로더를 `shuffle=True`로 두고 배치 평균을 냄 → 같은 모델의 점수가 채점마다 달라집니다.
5. seed 하나, 또는 기준선을 보지 않고 순위를 정함 → 테스트 그래프 112개에서 1개는 약 0.89%p입니다.

### 🏆 도전 과제
1. GIN의 리드아웃을 (a) 마지막 층 sum만 (b) 층별 mean concat (c) 층별 sum concat(지금)으로 바꿔 비교해 보세요 (강의 실습 2).
2. Cora GIN에 입력 드롭아웃 0.5를 넣거나 피처를 정규화(`T.NormalizeFeatures()`)해 정확도가 얼마나 오르는지 보세요.
3. PROTEINS 분할 seed를 11이 아닌 값으로 바꿔, 지금 본 순위가 분할에 얼마나 묶이는지 확인해 보세요.

### 교재 · 강의노트와 다른 점
- 이 노트북의 비교는 seed **3개**입니다. 강의노트 표는 seed **10개**(Cora GCN 80.21 ± 0.74, GIN 수정 55.90 ± 4.12, PROTEINS GIN 73.21 ± 2.66, GCN 69.38 ± 1.79)입니다.
- Cora GIN은 강의 벤치마크에서 한 번 고친 헤드(로짓에 BN/ReLU 없음)를 씁니다.
- 손실은 `log_softmax + nll_loss` 대신 짝이 맞는 `cross_entropy`(로짓 입력)를 썼습니다. 값은 같습니다.
